In [0]:
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

# 1. Definir el esquema (Best practice para evitar inferencia costosa)
schema = StructType([
    StructField("male", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("education", DoubleType(), True),
    StructField("currentSmoker", IntegerType(), True),
    StructField("cigsPerDay", DoubleType(), True),
    StructField("BPMeds", DoubleType(), True),
    StructField("prevalentStroke", IntegerType(), True),
    StructField("prevalentHyp", IntegerType(), True),
    StructField("diabetes", IntegerType(), True),
    StructField("totChol", DoubleType(), True),
    StructField("sysBP", DoubleType(), True),
    StructField("diaBP", DoubleType(), True),
    StructField("BMI", DoubleType(), True),
    StructField("heartRate", DoubleType(), True),
    StructField("glucose", DoubleType(), True),
    StructField("TenYearCHD", IntegerType(), True)
])

# 2. Configurar rutas (Ajustar según tu entorno)
landing_path = "/Volumes/workspace/default/tmp_landing/"
checkpoint_path = "/Volumes/workspace/default/tmp_landing/checkpoints/bronze_framingham"
table_name = "workspace.default.framingham_raw"

# 3. Leer usando Auto Loader
# cloudFiles se encarga de procesar solo archivos nuevos de forma incremental
df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("pathGlobFilter", "framingham*.csv")
    .option("header", "true")
    .schema(schema)
    .load(landing_path))

# 4. Enriquecer con metadatos (Esencial para auditoría en capa Bronze)
df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp()) \
                  .withColumn("source_file", col("_metadata.file_path"))

# 5. Escribir a la tabla Delta (Capa Bronze)
query = (df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True) # Procesa lo disponible y se detiene (ideal para batches)
    .toTable(table_name))

query.awaitTermination()